<a href="https://colab.research.google.com/github/lauraoliveiradesousa4-art/atividade-web/blob/main/AVALIA%C3%87%C3%83O_DE_DESEMPENHO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!pip install simpy

In [37]:
import random
import simpy

# Parâmetros da simulação
LAMBDA = 0.8
MU = 1.0
SERVIDORES = 1
N_CLIENTES = 500
SEMENTE = 2024001571
MOSTRAR_LOG = False

# Listas para armazenar os resultados
esperas = []
servicos = []
fila = []

# Simula a chegada, espera e atendimento de cada cliente
def cliente(env, nome, servidor, mu, log):
  chegada = env.now

  with servidor.request() as req:
    fila.append(len(servidor.queue))
    yield req

    w = env.now - chegada
    esperas.append(w)

    if log:
        print(f"{env.now:5.2f} {nome} inicia (esperou {w:.2f})")

    x = random.expovariate(mu)
    servicos.append(x)

    yield env.timeout(x)

# Gera as chegadas dos clientes em intervalos aleatorios
def chegadas(env, servidor, lam, mu, n, log):
      for i in range(n):
          yield env.timeout(random.expovariate(lam))
          env.process(cliente(env, f"c{i+1}", servidor, mu, log))

# Executa uma rodada completa da simulação
def rodar(lam=LAMBDA, mu=MU, servidores=SERVIDORES,
            n=N_CLIENTES, semente=SEMENTE, log=MOSTRAR_LOG):

    random.seed(semente)

    esperas.clear()
    servicos.clear()
    fila.clear()

    env = simpy.Environment()
    servidor = simpy.Resource(env, capacity=servidores)

    env.process(chegadas(env, servidor, lam, mu, n, log))

    env.run()

    ocupado = sum(servicos)
    total = env.now

    return ocupado, total

In [38]:
# Executa a rodada base com os parâmetros definidos
resultado = rodar()

print("Simulação executada com sucesso!")
print("Tempo ocupado:", resultado[0])
print("Tempo total:", resultado[1])

Simulação executada com sucesso!
Tempo ocupado: 501.52025194468115
Tempo total: 656.5775348934903


In [39]:
# Mostra o comportamento dos seis primeiros clientes
print("LOG DOS 6 PRIMEIROS CLIENTES")

rodar(n=6, semente=SEMENTE, log=True)

LOG DOS 6 PRIMEIROS CLIENTES
 0.02 c1 inicia (esperou 0.00)
 0.72 c2 inicia (esperou 0.34)
 0.88 c3 inicia (esperou 0.41)
 2.22 c4 inicia (esperou 1.28)
 2.76 c5 inicia (esperou 0.00)
 3.50 c6 inicia (esperou 0.65)


(3.6136120333233803, 4.119241096642136)

In [40]:
#Executa novamente a rodada base para recuperar os resultados dos 500 clientes
resultado = rodar()

print("Simulação base executada novamente!")
print("Tempo ocupado:", resultado[0])
print("Tempo total:", resultado[1])

Simulação base executada novamente!
Tempo ocupado: 501.52025194468115
Tempo total: 656.5775348934903


In [41]:
# Calcula as métricas principais da rodada base
rho = LAMBDA / MU
espera_media = sum(esperas) / N_CLIENTES
U = resultado[0] / resultado[1]
fila_maxima = max(fila)

print("MÉTRICAS DA RODADA BASE")
print(f"p (rho) = {LAMBDA} / {MU} = {rho: .3f}")
print(f"Espera média = {sum(esperas):.2f} / {N_CLIENTES} = {espera_media:.2f} min")
print(f"Ocupado = {resultado[0]:.2f} min")
print(f"Total = {resultado[1]:.2f} min")
print(f"U = {resultado[0]:.2f} / {resultado[1]:.2f} = {U:.3f} = {U*100:.1f}%")
print(f"Fila máxima = {fila_maxima} clientes")

MÉTRICAS DA RODADA BASE
p (rho) = 0.8 / 1.0 =  0.800
Espera média = 1081.61 / 500 = 2.16 min
Ocupado = 501.52 min
Total = 656.58 min
U = 501.52 / 656.58 = 0.764 = 76.4%
Fila máxima = 10 clientes


In [42]:
#Testa o efeito do aumento da taxa de chegada
print("TESTE 1 - VARIAÇÃO DE LAMBDA")

for lam in [0.50, 0.80, 0.95]:
    resultado = rodar(lam=lam, semente=SEMENTE)

    rho = lam / MU
    espera_media = sum(esperas) / N_CLIENTES
    U = resultado[0] / resultado[1]
    fila_maxima = max(fila)

    print(f"lambda = {lam:.2f}")
    print(f"rho = {rho:.3f}")
    print(f"Espera média = {espera_media:.2f} min")
    print(f"Utilização = {U:.3f} = {U*100:.1f}%")
    print(f"Fila máxima = {fila_maxima} clientes")


TESTE 1 - VARIAÇÃO DE LAMBDA
lambda = 0.50
rho = 0.500
Espera média = 0.83 min
Utilização = 0.500 = 50.0%
Fila máxima = 7 clientes
lambda = 0.80
rho = 0.800
Espera média = 2.16 min
Utilização = 0.764 = 76.4%
Fila máxima = 10 clientes
lambda = 0.95
rho = 0.950
Espera média = 10.35 min
Utilização = 0.985 = 98.5%
Fila máxima = 23 clientes


In [16]:
#Compara o desempenho usando um e dois servidores
print("TESTE 2 - NÚMERO DE SERVIDORES")

for numero_servidores in [1,2]:
    resultado = rodar(
        lam=1.2,
        mu=MU,
        servidores=numero_servidores,
        semente=SEMENTE
    )

    rho = 1.2 / numero_servidores
    espera_media = sum(esperas) / N_CLIENTES
    U = resultado[0] / (resultado[1] * numero_servidores)
    fila_maxima = max(fila)

    print(f"servidores = {numero_servidores}")
    print(f"rho = {rho:.3f}")
    print(f"Espera média = {espera_media:.2f} min")
    print(f"Utilização = {U:.3f} = {U*100:.1f}%")
    print(f"Fila máxima = {fila_maxima} cliente")

TESTE 2 - NÚMERO DE SERVIDORES
servidores = 1
rho = 1.200
Espera média = 36.11 min
Utilização = 1.000 = 100.0%
Fila máxima = 74 cliente
servidores = 2
rho = 0.600
Espera média = 0.59 min
Utilização = 0.608 = 60.8%
Fila máxima = 8 cliente


In [35]:
#Verifica se a mesma semente produz resultados repetíveis
print("TESTE 3 - REPETIBILIDADE")

resultado1 = rodar(lam=0.8, semente=SEMENTE)
espera1 = sum(esperas) / N_CLIENTES
fila1 = max(fila)

resultado2 = rodar(lam=0.8, semente=SEMENTE)
espera2 = sum(esperas) / N_CLIENTES
fila2 = max(fila)

print(f"Primeira execução:")
print(f"Espera média = {espera1:.2f} min")
print(f"Ocupado = {resultado[0]:.2f} min")
print(f"Total = {resultado[1]:.2f} min")
print(f"Fila máxima = {fila1} clientes")

print(f"Segunda execução:")
print(f"Espera média = {espera2:.2f} min")
print(f"Ocupado = {resultado[0]:.2f} min")
print(f"Total = {resultado[1]:.2f} min")
print(f"Fila máxima = {fila2} clientes")

print("Os resultados são iguais?",
      resultado1 == resultado2 and espera1 == espera2 and fila1 == fila2)

TESTE 3 - REPETIBILIDADE
Primeira execução:
Espera média = 2.16 min
Ocupado = 501.52 min
Total = 656.58 min
Fila máxima = 10 clientes
Segunda execução:
Espera média = 2.16 min
Ocupado = 501.52 min
Total = 656.58 min
Fila máxima = 10 clientes
Os resultados são iguais? True


 Conclusão

Os testes mostraram que o aumento da taxa de chegada provoca aumento da espera na fila.
A utilização do servidor permaneceu próxima do valor de rho nas simulações.
No Teste 2, a utilização de dois servidores reduziu a espera e resolveu o problema de sobrecarga.
O Teste 3 mostrou que a mesma semente produz resultados repetíveis.
Assim, o simulador passou pelos testes propostos.